In [ ]:
# Optional setup for the original database environment
# The public portfolio version does not download certificates automatically.
# See the connection cell below for secure local configuration.

In [ ]:
# The original course environment used an SSL certificate.
# In this public version, certificate paths should be supplied through
# the DB_SSLROOTCERT environment variable when required.

In [ ]:
# Required packages:
# pip install pandas sqlalchemy psycopg2-binary

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine

# Database credentials are intentionally not stored in this public notebook.
# To rerun the queries, define the following environment variables locally:
# DB_USER, DB_PASSWORD, DB_HOST, DB_PORT, DB_NAME
#
# If your database requires a CA certificate, also define DB_SSLROOTCERT.

required_vars = ['DB_USER', 'DB_PASSWORD', 'DB_HOST', 'DB_PORT', 'DB_NAME']
missing_vars = [name for name in required_vars if not os.getenv(name)]

if missing_vars:
    print(
        'Database connection is disabled in the public portfolio version. '
        'Set the required environment variables to rerun the SQL queries.'
    )
else:
    connection_string = (
        f"postgresql://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}"
        f"@{os.environ['DB_HOST']}:{os.environ['DB_PORT']}/{os.environ['DB_NAME']}"
    )

    connect_args = {}
    ssl_cert = os.getenv('DB_SSLROOTCERT')

    if ssl_cert:
        connect_args = {
            'sslmode': 'verify-full',
            'sslrootcert': ssl_cert
        }

    engine = create_engine(
        connection_string,
        connect_args=connect_args
    )

    print('Database engine created successfully.')


In [ ]:
query = '''
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
'''

if 'engine' in globals():
    try:
        tables = pd.read_sql(query, con=engine)
        print('Database connection successful.')
        display(tables)
    except Exception as error:
        print('Unable to connect to the database:')
        print(error)
else:
    print(
        'Connection not executed in the public portfolio version. '
        'The original query output is preserved below for review.'
    )

# 📚 SQL Business Analysis — Digital Book Service

## Introduction

Changes in consumer behaviour created new opportunities for digital reading products and services. In this project, I analyse the database of a fictional book service to understand both the catalogue and the way readers interact with it.

The database contains information about:

- books;
- authors;
- publishers;
- user ratings;
- written reviews.

## Objective

The goal is to use SQL to answer a set of business questions that could support the development of a new reading-related product.

The analysis investigates:

1. how many books were published after 1 January 2000;
2. the number of ratings, written reviews, and average rating for each book;
3. the publisher with the largest number of books longer than 50 pages;
4. the author with the highest average rating among books with at least 50 ratings;
5. the average number of written reviews produced by users who rated more than 50 books.

SQL is used for the analytical logic, while pandas is used only to display query results inside the notebook.


## 1. Database Exploration

Before answering the business questions, I explore the structure of the database.

The analysis focuses on five main tables:

- `books` — book-level catalogue information;
- `authors` — author information;
- `publishers` — publisher information;
- `ratings` — user ratings;
- `reviews` — written user reviews.

The `books` table is the central catalogue table. It connects to `authors` through `author_id` and to `publishers` through `publisher_id`. Both `ratings` and `reviews` connect to books through `book_id`.

The first step is to preview each table and understand its structure.


In [112]:
query = """
SELECT *
FROM books
LIMIT 5;
"""

books_sample = pd.read_sql(query, con=engine)
books_sample

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


In [113]:
query = """
SELECT *
FROM authors
LIMIT 5;
"""

authors_sample = pd.read_sql(query, con=engine)
authors_sample

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


In [114]:
query = """
SELECT *
FROM publishers
LIMIT 5;
"""

publishers_sample = pd.read_sql(query, con=engine)
publishers_sample

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


In [115]:
query = """
SELECT *
FROM ratings
LIMIT 5;
"""

ratings_sample = pd.read_sql(query, con=engine)
ratings_sample

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


In [116]:
query = """
SELECT *
FROM reviews
LIMIT 5;
"""

reviews_sample = pd.read_sql(query, con=engine)
reviews_sample

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


### Relational Structure

The main relationships are:

- `authors.author_id` → `books.author_id`
- `publishers.publisher_id` → `books.publisher_id`
- `books.book_id` → `ratings.book_id`
- `books.book_id` → `reviews.book_id`

The `ratings` and `reviews` tables also contain `username`, which allows user activity to be explored across both forms of interaction.


In [117]:
query = """
SELECT 'books' AS table_name, COUNT(*) AS row_count
FROM books

UNION ALL

SELECT 'authors' AS table_name, COUNT(*) AS row_count
FROM authors

UNION ALL

SELECT 'publishers' AS table_name, COUNT(*) AS row_count
FROM publishers

UNION ALL

SELECT 'ratings' AS table_name, COUNT(*) AS row_count
FROM ratings

UNION ALL

SELECT 'reviews' AS table_name, COUNT(*) AS row_count
FROM reviews;
"""

table_counts = pd.read_sql(query, con=engine)
table_counts

,table_name,row_count
0,books,1000
1,authors,636
2,publishers,340
3,ratings,6456
4,reviews,2793


### Initial Data Overview

The database combines catalogue information with user-generated activity.

It contains:

- **1,000 books**
- **636 authors**
- **340 publishers**
- **6,456 ratings**
- **2,793 written reviews**

This structure makes it possible to analyse both catalogue characteristics and reader engagement.


## 2. Books Published After 1 January 2000

The first question asks how many books in the catalogue were published after 1 January 2000.

Because the requirement is strictly *after* that date, the SQL filter uses:

`publication_date > DATE '2000-01-01'`


In [118]:

query = """
SELECT
    COUNT(*) AS books_after_2000
FROM books
WHERE publication_date > DATE '2000-01-01';
"""

books_after_2000 = pd.read_sql(query, con=engine)
books_after_2000


,books_after_2000
0,819


### Finding

The query identified **819 books published after 1 January 2000**.

This suggests that the catalogue has a strong representation of more recent titles, which could be relevant for a product focused on modern reading preferences and content discovery.


## 3. Ratings, Reviews and Average Rating by Book

The next analysis compares several forms of reader interaction for every book:

- number of ratings;
- number of written reviews;
- average rating.

Because a book can have multiple ratings and multiple reviews, I aggregate these two tables separately before joining them to `books`. This avoids the row-multiplication problem that would occur if `ratings` and `reviews` were joined directly before aggregation.

I use `LEFT JOIN` so that all catalogue books remain in the result, even if a title has no written reviews.


In [119]:
query = """
WITH review_stats AS (
    SELECT
        book_id,
        COUNT(review_id) AS review_count
    FROM reviews
    GROUP BY book_id
),

rating_stats AS (
    SELECT
        book_id,
        COUNT(rating_id) AS rating_count,
        AVG(rating) AS avg_rating
    FROM ratings
    GROUP BY book_id
)

SELECT
    b.book_id,
    b.title,
    COALESCE(rts.rating_count, 0) AS rating_count,
    COALESCE(rs.review_count, 0) AS review_count,
    ROUND(rts.avg_rating, 2) AS avg_rating
FROM books AS b
LEFT JOIN rating_stats AS rts
    ON b.book_id = rts.book_id
LEFT JOIN review_stats AS rs
    ON b.book_id = rs.book_id
ORDER BY review_count DESC, avg_rating DESC, b.title;
"""

book_stats = pd.read_sql(query, con=engine)
book_stats

,book_id,title,rating_count,review_count,avg_rating
0,948,Twilight (Twilight #1),160,7,3.66
1,302,Harry Potter and the Prisoner of Azkaban (Harr...,82,6,4.41
2,299,Harry Potter and the Chamber of Secrets (Harry...,80,6,4.29
3,656,The Book Thief,53,6,4.26
4,734,The Glass Castle,29,6,4.21
...,...,...,...,...,...
995,191,Disney's Beauty and the Beast (A Little Golden...,1,0,4.00
996,221,Essential Tales and Poems,3,0,4.00
997,387,Leonardo's Notebooks,2,0,4.00
998,83,Anne Rice's The Vampire Lestat: A Graphic Novel,3,0,3.67


### Finding

The query returned information for all **1,000 books** in the catalogue.

**Twilight (Twilight #1)** had the largest number of written reviews, with **7**, and an average rating of **3.66**.

Some books with fewer reviews had higher average ratings. For example, **Harry Potter and the Prisoner of Azkaban** had **6 reviews** and an average rating of **4.41**.

This highlights why ratings and written reviews are useful complementary measures: one captures numerical sentiment, while the other reflects a deeper level of reader interaction.


## 4. Publisher with the Most Books Longer Than 50 Pages

This analysis identifies the publisher responsible for the largest number of books with more than 50 pages.

The page-count condition helps exclude very short publications and keeps the focus on more substantial titles.

The query joins `books` and `publishers` through `publisher_id`.


In [120]:
query = """
SELECT
    p.publisher,
    COUNT(b.book_id) AS book_count
FROM publishers AS p
INNER JOIN books AS b
    ON p.publisher_id = b.publisher_id
WHERE b.num_pages > 50
GROUP BY
    p.publisher_id,
    p.publisher
ORDER BY book_count DESC
LIMIT 1;
"""

top_publisher = pd.read_sql(query, con=engine)
top_publisher

,publisher,book_count
0,Penguin Books,42


### Finding

**Penguin Books** was the publisher with the largest number of books longer than 50 pages, with **42 titles**.

This indicates a strong presence in the catalogue when focusing on longer-form publications.


## 5. Highest-Rated Author Among Books with at Least 50 Ratings

To make the comparison more reliable, I first identify books with at least 50 user ratings.

I then calculate the author's average rating using the individual rating records associated with those qualified books.

The minimum-rating threshold helps reduce the influence of books with only a small number of ratings.


In [121]:
query = """
WITH qualified_books AS (
    SELECT
        book_id
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating_id) >= 50
)

SELECT
    a.author,
    ROUND(AVG(r.rating), 3) AS avg_rating
FROM authors AS a
INNER JOIN books AS b
    ON a.author_id = b.author_id
INNER JOIN ratings AS r
    ON b.book_id = r.book_id
INNER JOIN qualified_books AS qb
    ON b.book_id = qb.book_id
GROUP BY
    a.author_id,
    a.author
ORDER BY avg_rating DESC
LIMIT 1;
"""

top_author = pd.read_sql(query, con=engine)
top_author

,author,avg_rating
0,J.K. Rowling/Mary GrandPré,4.287


### Finding

Among authors associated with books that received at least 50 ratings, **J.K. Rowling/Mary GrandPré** had the highest average rating, at approximately **4.287**.

Because the result is based on titles with a meaningful volume of user feedback, it provides a stronger signal than an average based on only a few ratings.


## 6. Review Activity Among Highly Active Raters

The final analysis focuses on highly active users.

First, I identify users who rated more than 50 distinct books. I then calculate how many written reviews those users produced and take the average across the group.


In [122]:
query = """
WITH active_users AS (
    SELECT
        username
    FROM ratings
    GROUP BY username
    HAVING COUNT(DISTINCT book_id) > 50
),

review_counts AS (
    SELECT
        r.username,
        COUNT(r.text) AS review_count
    FROM reviews AS r
    INNER JOIN active_users AS au
        ON r.username = au.username
    GROUP BY r.username
)

SELECT
    ROUND(AVG(review_count), 2) AS avg_review_count
FROM review_counts;
"""

avg_reviews_active_users = pd.read_sql(query, con=engine)
avg_reviews_active_users

,avg_review_count
0,24.33


### Finding

Users who rated more than 50 books wrote an average of approximately **24.33 reviews per user**.

This suggests that highly active raters also contribute meaningfully to written content, although writing a review requires more effort than simply assigning a numerical rating.


## Overall Conclusion

This project used SQL to explore both catalogue characteristics and reader engagement within a digital book service.

The main findings were:

- **819 books** were published after 1 January 2000;
- all **1,000 books** were analysed using rating count, review count, and average rating;
- **Twilight (Twilight #1)** had the highest number of written reviews, with **7**;
- **Penguin Books** had the largest number of books longer than 50 pages, with **42 titles**;
- **J.K. Rowling/Mary GrandPré** had the highest average rating among authors whose qualifying books received at least 50 ratings, at approximately **4.287**;
- users who rated more than 50 books wrote an average of approximately **24.33 reviews**.

From a business perspective, the results show that catalogue information becomes more useful when combined with reader behaviour.

Books and authors that combine strong ratings with meaningful interaction volumes could support recommendation and discovery features. Highly active readers may also be an important audience for community features designed to encourage reviews and deeper engagement.

## What I Learned

What I enjoyed most in this project was using SQL to move from a relational database to clear business insights.

The project strengthened my understanding of joins, CTEs, aggregation, `GROUP BY`, `HAVING`, `COUNT(DISTINCT ...)`, and the importance of structuring queries carefully to avoid duplicated results.

It also reinforced something I find especially valuable in analytics: a metric becomes more meaningful when it is considered in context. An average rating alone tells only part of the story — the number of ratings, reviews, and level of user engagement help make that information much more useful for decision-making.
